# M5 · Backfill `fact_stop`

**Goal:** load the full history of `staging.stop` into `warehouse.fact_stop` in 30-day chunks.

**SQL file:** `sql/12_fact_stop_incremental.sql` (parameters: `:window_start`, `:window_end`, `:etl_run_id`).

**Exit criterion:**
- `warehouse.etl_watermark.last_event_time` for `fact_stop` ≥ `MAX(staging.stop.stop_start)`.
- Row count in the fact table looks right (orders of magnitude vs staging).
- Validation V1 (row count) passes.

> **Run order:** this notebook assumes the facts before it (lower number) have already been loaded.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — config, SQL, and the backfill plan

In [2]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(stop_start) FROM staging.stop")).scalar_one()
print('staging max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('12_fact_stop_incremental.sql')[:600], '...')


chunk_days = 30   min_time = 2019-10-01 00:00:00
staging max event-time = 2026-04-10 13:28:49

----- SQL file preview -----
-- =============================================================================
-- 12_fact_stop_incremental.sql
-- =============================================================================
-- Stops. Applies C1 (temporal) and C6 (duration bounds). Derives stop_type
-- from duration bucket.
-- =============================================================================

CREATE TABLE IF NOT EXISTS warehouse.fact_stop (
  stop_sk           BIGSERIAL PRIMARY KEY,
  tenant_id         INTEGER NOT NULL,
  device_id         BIGINT NOT NULL,
  stop_start        TIMESTAMP NOT NULL,
  stop_end       ...


## 3. Execute — loop 30-day chunks, call `load_fact_incremental` for each

In [3]:
import time, pandas as pd
from accent_fleet.transforms.facts import load_fact_incremental
from accent_fleet.pipeline.run_log import begin_run, end_run

run_id = begin_run(mode='notebook:03_load_fact_stop')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_stop', run_id=run_id, window_end=next_cursor)
        progress.append({
            'window_start': cursor, 'window_end': next_cursor,
            'rows_loaded': res.rows_loaded, 'new_max': res.new_max_event_time,
        })
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise
print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


etl_run_id = 15
2026-04-25 02:59:59 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 31, 0, 0) fact=fact_stop run_id=15 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-25 03:00:02 [info     ] fact_load.done                 fact=fact_stop new_watermark=datetime.datetime(2019, 10, 31, 0, 0) rows=128399
2026-04-25 03:00:02 [info     ] fact_load.start                end=datetime.datetime(2019, 11, 30, 0, 0) fact=fact_stop run_id=15 start=datetime.datetime(2019, 10, 30, 23, 50)
2026-04-25 03:00:07 [info     ] fact_load.done                 fact=fact_stop new_watermark=datetime.datetime(2019, 11, 30, 0, 0) rows=198915
2026-04-25 03:00:08 [info     ] fact_load.start                end=datetime.datetime(2019, 12, 30, 0, 0) fact=fact_stop run_id=15 start=datetime.datetime(2019, 11, 29, 23, 50)
2026-04-25 03:00:13 [info     ] fact_load.done                 fact=fact_stop new_watermark=datetime.datetime(2019, 12, 30, 0, 0) rows=207061
2026-04-25 03:00:14 [info     ] f

,window_start,window_end,rows_loaded,new_max
70,2025-07-01,2025-07-31 00:00:00,176454,2025-07-31 00:00:00
71,2025-07-31,2025-08-30 00:00:00,171443,2025-08-30 00:00:00
72,2025-08-30,2025-09-29 00:00:00,165348,2025-09-29 00:00:00
73,2025-09-29,2025-10-29 00:00:00,166647,2025-10-29 00:00:00
74,2025-10-29,2025-11-28 00:00:00,157104,2025-11-28 00:00:00
75,2025-11-28,2025-12-28 00:00:00,150931,2025-12-28 00:00:00
76,2025-12-28,2026-01-27 00:00:00,133038,2026-01-27 00:00:00
77,2026-01-27,2026-02-26 00:00:00,148839,2026-02-26 00:00:00
78,2026-02-26,2026-03-28 00:00:00,128714,2026-03-28 00:00:00
79,2026-03-28,2026-04-10 13:28:49,69729,2026-04-10 13:28:49


## 4. Inspect — watermark, counts, quarantine

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_stop'"), conn)
    fc = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_stop')).scalar_one()
    sc = conn.execute(text('SELECT COUNT(*) FROM staging.stop')).scalar_one()
    q  = pd.read_sql(text("""SELECT rule_id, COUNT(*) AS n
                               FROM warehouse.quarantine_rejected
                               WHERE source_table='stop' AND etl_run_id = :rid
                               GROUP BY rule_id"""), conn, params={'rid': run_id})
print(f'staging.stop: {sc:,} rows')
print(f'warehouse.fact_stop: {fc:,} rows')
display(wm); display(q)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — backfill of fact_stop complete.')


staging.stop: 12,133,639 rows
warehouse.fact_stop: 11,472,177 rows


,layer,table_name,last_event_time,last_run_at,last_etl_run_id,rows_loaded_total,notes
0,warehouse,fact_stop,2026-04-10 13:28:49,2026-04-25 02:06:59.410287+00:00,15,11472541,Incremental on stop_start


,rule_id,n


OK — backfill of fact_stop complete.
